# Azure DevOps Features Analysis

This notebook reads the Azure DevOps backlog data and creates a comprehensive DataFrame showing all features with their characteristics.

## 1. Import libraries and read data
### Libraries

In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import date

### Refresh Backlog Data
Fetch the latest data from Azure DevOps and transform it into a hierarchical structure.  

Runtime ~ 7,5 min

In [2]:
%run ../src/scripts/fetch_dena_backlog.py
%run ../src/scripts/backlog_hierarchical_transformer.py

AZURE DEVOPS BACKLOG DATA FETCHER - DENA

✓ API key loaded from .env file
Organization: HogeschoolUtrecht
Fetching all projects from Azure DevOps organization...
  URL: https://dev.azure.com/HogeschoolUtrecht/_apis/projects?api-version=7.1
  Attempt 1/3...
✓ Found 3 project(s)

Projects found:
  - Dienst IM ICT (ID: 175e4ac1-3967-433d-875a-b00159276613)
  - DevOps DenA (ID: 8b224685-8bd1-4a9a-9650-c62d60c23e77)
  - Data Science Pool (ID: 63c0199a-9442-4f4e-9679-e983b253b63c)

✓ Found project: DevOps DenA
  Project ID: 8b224685-8bd1-4a9a-9650-c62d60c23e77

Focusing on: DevOps DenA

FETCHING WORK ITEMS FROM ALL PROJECTS

Fetching work items from project: DevOps DenA
Attempting to fetch work items using simplified API...
  Attempt 1/3 - Querying work item IDs...
✓ Found 15749 work items in DevOps DenA
  Fetching details in 315 batch(es)...
  Fetching batch 1/315 (50 items)...
    ✓ Retrieved 50 items from batch 1
  Fetching batch 2/315 (50 items)...
    ✓ Retrieved 50 items from batch 2
 

## 2. Load JSON Data

In [3]:
# Define the path to the JSON file using today's date
today = date.today().strftime('%Y%m%d')
data_file = Path(f'../azure_devops_data/backlog_simplified_{today}.json')

# Load the JSON data
with open(data_file, 'r', encoding='utf-8') as f:
    backlog_data = json.load(f)

# Display metadata
print("Metadata:")
print(f"Organization: {backlog_data['metadata']['organization']}")
print(f"Total work items: {backlog_data['metadata']['total_work_items']}")
print(f"Data fetched at: {backlog_data['metadata']['source_fetched_at']}")
print("\nStatistics by type:")
for work_type, count in backlog_data['statistics']['by_type'].items():
    print(f"  {work_type}: {count}")

Metadata:
Organization: HogeschoolUtrecht
Total work items: 15749
Data fetched at: 2026-10-02T10:19:37.997944

Statistics by type:
  Product Backlog Item: 4021
  ETL: 362
  Universe: 167
  Feature: 626
  Task: 8781
  Rapport: 537
  Bug: 424
  Epic: 95
  R: 59
  DM: 610
  Test Case: 2
  Python: 65


Exclusie/inclusie criteria:
- All features from 20 february 2025 onwards

## 3. Gather Data on Feature level
### Get all Features and their parent Epic information
- epic name
- feature name
- date last changed (= closing date)
- area
- Moeite Int (Effort DenA)
- Business Value

In [4]:
# Transform the hierarchical backlog data into a flat structure on a feature level into a df
def flatten_backlog(data):
    """Flatten hierarchical backlog items into a list."""
    flattened = []

    def visit(item, parents=None):
        parents = parents or []
        record = item.copy()

        record["parent_id"] = parents[-1]["id"] if parents else None
        record["parent_title"] = parents[-1].get("title") if parents else None

        children = record.pop("children", [])
        flattened.append(record)

        for child in children:
            visit(child, parents + [record])

    for item in data.get("backlog", []):
        visit(item)

    return flattened

flat_backlog = flatten_backlog(backlog_data)

# Convert to dataframe
df_flat_backlog = pd.DataFrame(flat_backlog)

# Extract only features and their parent epic information
df_features = df_flat_backlog[df_flat_backlog['type'] == 'Feature'].copy()


## 4. Process Data

## 4.1 Exclude features closed before 20 february 2025

In [5]:
# Include only features closed after 2025-02-20
df_features_closed = df_features[df_features['closed_date'] >= '2025-02-20'].copy()

# Print how many features were excluded
excluded_count = len(df_features) - len(df_features_closed)
print(f"Excluded {excluded_count} features closed before 2025-02-20.")
print(f"Remaining features: {len(df_features_closed)}")

Excluded 475 features closed before 2025-02-20.
Remaining features: 127


## 4.2 Clean data

In [6]:
# Convert effort into an integer
df_features_closed['moeite_int'] = pd.to_numeric(
    df_features_closed['moeite_int'],
    errors='coerce'
).astype('Int64')

In [7]:
# Adjust Area Path to only exclude the first level (e.g., "Area1\Area2" becomes "Area2")
df_features_closed['area_path'] = df_features_closed['area_path'].apply(lambda x: '\\'.join(x.split('\\')[1:]) if pd.notnull(x) and '\\' in x else x)

In [8]:
# Convert closed date into date
# Use format='ISO8601' to correctly parse mixed timestamp formats (with/without milliseconds)
df_features_closed['closed_date'] = pd.to_datetime(df_features_closed['closed_date'], errors='coerce', format='ISO8601').dt.date

### 4.3 Enrich data


In [9]:
# Add a new column that indicates whether the feature area is D&A or not (hard-coded)
df_features_closed['is_DA'] = df_features_closed['area_path'].apply(lambda x: any(keyword in x for keyword in ['AI','DenA', 'DenA Architectuur', 'DevOps DenA', 'Domeinteams']) if pd.notnull(x) else False)

# 5. Get additional state information for each feature
For each feature, retrieve the state updates from DevOps REST API and add them to the DataFrame. This will include:
- WorkItemID
- PreviousState
- NewState
- ChangedDate

## 5.1 Get state updates for each feature from API
Runtime ~ 1,5 min

In [10]:
# Create a list of unique feature IDs for further processing
feature_ids = df_features_closed['id'].tolist()
feature_ids_args = ' '.join(str(int(fid)) for fid in feature_ids)

# Fetch the latest state updates using the fetch_state_updates.py script
%run ../src/scripts/fetch_work_item_updates.py {feature_ids_args}

# Define the path to the JSON file using today's date
today = date.today().strftime('%Y%m%d')
data_file = Path(f'../azure_devops_data/work_item_updates_{today}.json')

# Load the JSON data
with open(data_file, 'r', encoding='utf-8') as f:
    feature_update_data = json.load(f)

AZURE DEVOPS WORK ITEM UPDATES FETCHER
Organization: HogeschoolUtrecht
Project: DevOps DenA
Work Item IDs: [76550, 120228, 92477, 118036, 128282, 111827, 126979, 123848, 120073, 120240, 114622, 113423, 110648, 127137, 106708, 79592, 111822, 112070, 99594, 114697, 104444, 102380, 89663, 105506, 101148, 92469, 106434, 107700, 74616, 118276, 75164, 107923, 123712, 109826, 109820, 128895, 85109, 105549, 110634, 130147, 115790, 123753, 133518, 126183, 113339, 118960, 123734, 121457, 154237, 124160, 158043, 114477, 104018, 91579, 84870, 143971, 111666, 157599, 113550, 111668, 100379, 160320, 136823, 151810, 118648, 134980, 134174, 142345, 129492, 83512, 78298, 110115, 113863, 109976, 149428, 152557, 92583, 88319, 141222, 84511, 163293, 139423, 138565, 130666, 85252, 83377, 108982, 121340, 128546, 128433, 150514, 104080, 97731, 130556, 131285, 106477, 117125, 139126, 130884, 124786, 150913, 133057, 140740, 111698, 161885, 145651, 155621, 139386, 130198, 120042, 138328, 98252, 146929, 155622, 

## 5.2 Transform into df

In [11]:
# Create one row for every feature state transition.
def flatten_feature_updates(data):
    """Return feature state transitions from Azure DevOps update data."""
    rows = []

    for work_item_id, updates in data.get("updates", {}).items():
        for update in updates:
            state_change = update.get("fields", {}).get("System.State")
            if not state_change:
                continue

            # Use fields.System.ChangedDate (actual change moment) instead of revisedDate,
            # which is set to the sentinel "9999-01-01" for a work item's current/latest revision.
            changed_date = update.get("fields", {}).get("System.ChangedDate", {}).get("newValue")

            rows.append({
                "work_item_id": int(update.get("workItemId", work_item_id)),
                "revision": update.get("rev"),
                "changed_date": changed_date,
                "previous_state": state_change.get("oldValue"),
                "new_state": state_change.get("newValue"),
                "changed_by": update.get("revisedBy", {}).get("displayName"),
            })

    return rows

flat_feature_updates = flatten_feature_updates(feature_update_data)
df_feature_state_updates = pd.DataFrame(flat_feature_updates)
df_feature_state_updates["changed_date"] = pd.to_datetime(
    df_feature_state_updates["changed_date"], utc=True, errors="coerce", format="ISO8601"
)
df_feature_state_updates = df_feature_state_updates.sort_values(
    ["work_item_id", "changed_date"]
).reset_index(drop=True)

df_feature_state_updates

,work_item_id,revision,changed_date,previous_state,new_state,changed_by
0,74616,1,2023-08-18 12:35:35.960000+00:00,None,New,Jeroen van Ravensteijn
1,74616,19,2024-05-17 15:45:01.793000+00:00,New,In Progress,Klaske de Hoop
2,74616,23,2024-06-11 08:21:27.720000+00:00,In Progress,Ingediend,Jeroen van Ravensteijn
3,74616,24,2024-07-02 08:16:19.917000+00:00,Ingediend,New,Jeroen van Ravensteijn
4,74616,31,2024-08-22 11:57:28.553000+00:00,New,In Progress,Fraukje Coopmans
...,...,...,...,...,...,...
552,163293,1,2026-08-06 09:57:45.860000+00:00,None,New,Anne Leemans
553,163293,2,2026-08-06 10:04:41.260000+00:00,New,In Progress,Anne Leemans
554,163293,4,2026-08-14 08:44:06.007000+00:00,In Progress,Done,Anne Leemans
555,163293,8,2026-08-24 06:17:24.110000+00:00,Done,In Progress,Anne Leemans


In [12]:
# Order the df by work_item_id and changed_date
df_feature_state_updates = df_feature_state_updates.sort_values(["work_item_id", "changed_date"]).reset_index(drop=True)

## 5.3 Add calculated columns with insights
 

### 5.3.1 Wachttijd_feature
For a feature, calculate the date difference between a new_state = "Geprioriteerd" row and a row with prev_state = "Geprioriteerd" and new_state = "In Progress". This will give the waiting time for a feature to be started after being prioritized.

First check:
- how many features have been prioritized (at least once)?

In [13]:
# For each feature count the number of Geprioriteerd states, including features with no "Geprioriteerd" states (which will be counted as 0)
geprioriteerd_counts = df_feature_state_updates.groupby('work_item_id').apply(lambda x: (x['new_state'] == 'Geprioriteerd').sum())
print("Number of features with 'Geprioriteerd' states:")
print(geprioriteerd_counts.value_counts().sort_index())

Number of features with 'Geprioriteerd' states:
0    85
1    38
2     3
3     1
Name: count, dtype: int64


C:\Users\fraukje.coopmans\AppData\Local\Temp\ipykernel_16352\1799019949.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  geprioriteerd_counts = df_feature_state_updates.groupby('work_item_id').apply(lambda x: (x['new_state'] == 'Geprioriteerd').sum())


Looking at the features that have no prioritized state, are there any features from a Domeinteam (defined by the Area). 

In [14]:
# Find the feature IDs that have no "Geprioriteerd" states, and look them up in the df_features_closed to see if they belong to a Domeinteam (defined by the Area).
features_no_geprioriteerd = geprioriteerd_counts[geprioriteerd_counts == 0].index.tolist()

# Look up the features in df_features_closed
features_no_geprioriteerd_df = df_features_closed[df_features_closed['id'].isin(features_no_geprioriteerd)]

# Count the number of features per area_path
print("Features with no 'Geprioriteerd' states and their Area Paths:")
area_counts = features_no_geprioriteerd_df['area_path'].value_counts()
print(area_counts)

Features with no 'Geprioriteerd' states and their Area Paths:
area_path
DenA Architectuur      26
DevOps DenA            17
Domeinteams            11
AI                      9
Data Governance         6
Education Analytics     6
Personeel               3
DenA                    2
Bedrijfsvoering         2
Research Analytics      1
Data Science Pool       1
Finance                 1
Name: count, dtype: int64


In [15]:
# For each feature count the number of Ingediend states, including features with no "Ingediend" states (which will be counted as 0)
ingediend_counts = df_feature_state_updates.groupby('work_item_id').apply(lambda x: (x['new_state'] == 'Ingediend').sum())
print("Number of features with 'Ingediend' states:")
print(ingediend_counts.value_counts().sort_index())

Number of features with 'Ingediend' states:
0    63
1    57
2     4
3     3
Name: count, dtype: int64


C:\Users\fraukje.coopmans\AppData\Local\Temp\ipykernel_16352\3309831365.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ingediend_counts = df_feature_state_updates.groupby('work_item_id').apply(lambda x: (x['new_state'] == 'Ingediend').sum())


### Bereken wachttijd_feature in dagen

In [16]:
# For a feature, calculate the date difference between a new_state = "Geprioriteerd" row and a row with 
# prev_state = "Geprioriteerd" and new_state = "In Progress". This will give the waiting time for a feature 
# to be started after being prioritized.
# If a feature has no prev_state 'Geprioriteerd' it should be NA. If a feature has multiple prev_state 
# 'Geprioriteerd' with new_state = 'In progress' it should take the first one.
def calculate_waiting_time(df):
    waiting_times = []

    for feature_id, group in df.groupby('work_item_id'):
        group = group.sort_values('changed_date')
        geprioriteerd_rows = group[group['new_state'] == 'Geprioriteerd']
        in_progress_rows = group[(group['previous_state'] == 'Geprioriteerd') & (group['new_state'] == 'In Progress')]

        if not geprioriteerd_rows.empty and not in_progress_rows.empty:
            first_geprioriteerd_date = geprioriteerd_rows.iloc[0]['changed_date']
            first_in_progress_date = in_progress_rows.iloc[0]['changed_date']
            waiting_time = (first_in_progress_date - first_geprioriteerd_date).days
            waiting_times.append({'work_item_id': feature_id, 'waiting_time_days': waiting_time})
        else:
            waiting_times.append({'work_item_id': feature_id, 'waiting_time_days': None})

    return pd.DataFrame(waiting_times)

calculated_waiting_times = calculate_waiting_time(df_feature_state_updates)

# Merge the waiting times with the features dataframe to get the area_path and other details
df_features_with_waiting_time = df_features_closed.merge(calculated_waiting_times, left_on='id', right_on='work_item_id', how='left')

### 5.3.2 Onderhanden_werk_feature in dagen

In [17]:
# Check features with In Progress state without a prior Geprioriteerd state
features_in_progress_without_geprioriteerd = df_features_with_waiting_time[df_features_with_waiting_time['waiting_time_days'].isnull() & (df_features_with_waiting_time['id'].isin(df_feature_state_updates[df_feature_state_updates['new_state'] == 'In Progress']['work_item_id']))]
print("Features with 'In Progress' state but no prior 'Geprioriteerd' state:")
print(features_in_progress_without_geprioriteerd[['id', 'title', 'area_path', 'closed_date']])

Features with 'In Progress' state but no prior 'Geprioriteerd' state:
         id                                              title  \
0     76550  Aantallen Minoren inschrijvingen (t.b.v. Onder...   
1    120228  Azure Platform (ADF) kosten/ reservations inzi...   
2     92477  ADF: Architectuur verbeteringen Blok C (colleg...   
4    128282  ADF: Architectuur verbeteringen Blok E (colleg...   
5    111827  DevOps inrichting aanpassen samenwerken tussen...   
..      ...                                                ...   
120  142267           Power BI chatbot met NSE open antwoorden   
121  142591      AI kennisdeling & inspiratie [jan-maart 2026]   
122  153619        AI kennisdeling & inspiratie [apr-jul 2026]   
123  138894           Prototype Dataflow chatbot ("Ask AInne")   
124  150199  Plan van Aanpak Dataportaal (CKAN) op HU omgeving   

               area_path closed_date  
0    Education Analytics  2025-03-28  
1            DevOps DenA  2025-05-19  
2      DenA Architec

In [18]:
# For a feature, calculate the date difference between a new_state = "In Progress" row and a row with 
# prev_state = "In Progress" and new_state = "Done". This will give the time a feature has been worked on. 

def calculate_onderhanden_werk(df):
    onderhanden_werk_times = []

    for feature_id, group in df.groupby('work_item_id'):
        group = group.sort_values('changed_date')
        in_progress_rows = group[group['new_state'] == 'In Progress']
        done_rows = group[(group['previous_state'] == 'In Progress') & (group['new_state'] == 'Done')]

        total_onderhanden_werk_time = 0

        for _, in_progress_row in in_progress_rows.iterrows():
            corresponding_done_row = done_rows[done_rows['changed_date'] > in_progress_row['changed_date']]
            if not corresponding_done_row.empty:
                first_done_date = corresponding_done_row.iloc[0]['changed_date']
                onderhanden_werk_time = (first_done_date - in_progress_row['changed_date']).days
                total_onderhanden_werk_time += onderhanden_werk_time

        if total_onderhanden_werk_time > 0:
            onderhanden_werk_times.append({'work_item_id': feature_id, 'onderhanden_werk_days': total_onderhanden_werk_time})
        else:
            onderhanden_werk_times.append({'work_item_id': feature_id, 'onderhanden_werk_days': None})

    return pd.DataFrame(onderhanden_werk_times)

calculated_onderhanden_werk = calculate_onderhanden_werk(df_feature_state_updates)

# Merge with the features dataframe
df_features_closed_final = df_features_with_waiting_time.merge(calculated_onderhanden_werk, on='work_item_id', how='left')

Bovenstaande kolom nog testen! Vervolgens joinen op feature df

### 5.3.3 Rounded_effort_closed_per_month

In [19]:
df_features_closed_final['year_month'] = df_features_closed_final['closed_date'].apply(lambda x: x.strftime('%Y-%m') if pd.notnull(x) else None)
df_features_closed_final['summed_closed_effort_per_year_month'] = df_features_closed_final.groupby('year_month')['moeite_int'].transform('sum')

## 5.4 Calculate one-off values

### 5.4.1 Mean onderhanden werk

In [20]:
df_features_closed_final.head()

,id,type,title,state,assigned_to,created_date,closed_date,priority,value_area,moeite_int,...,parent_id,parent_title,start_date,target_date,is_DA,work_item_id,waiting_time_days,onderhanden_werk_days,year_month,summed_closed_effort_per_year_month
0,76550,Feature,Aantallen Minoren inschrijvingen (t.b.v. Onder...,Done,Unassigned,2023-09-15T07:15:51.207Z,2025-03-28,2.0,Business,<NA>,...,72706.0,Vraagstukken Onderwijs (noodzakelijk vanuit ar...,NaN,NaN,False,76550,NaN,NaN,2025-03,4
1,120228,Feature,Azure Platform (ADF) kosten/ reservations inzi...,Done,Unassigned,2025-04-08T13:47:02.06Z,2025-05-19,2.0,Business,<NA>,...,75166.0,D&A: ADF Architectuur verbeteren (2024-2025),NaN,NaN,True,120228,NaN,40.0,2025-05,4
2,92477,Feature,ADF: Architectuur verbeteringen Blok C (colleg...,Done,Unassigned,2024-04-19T11:06:52.87Z,2025-04-08,2.0,Business,<NA>,...,75166.0,D&A: ADF Architectuur verbeteren (2024-2025),NaN,NaN,True,92477,NaN,62.0,2025-04,2
3,118036,Feature,ADF: Architectuur verbeteringen Blok D (colleg...,Done,Unassigned,2025-03-19T09:35:31.447Z,2025-08-18,2.0,Business,<NA>,...,75166.0,D&A: ADF Architectuur verbeteren (2024-2025),NaN,NaN,True,118036,NaN,NaN,2025-08,1
4,128282,Feature,ADF: Architectuur verbeteringen Blok E (colleg...,Done,Unassigned,2025-07-10T10:48:55.177Z,2025-08-26,2.0,Business,<NA>,...,75166.0,D&A: ADF Architectuur verbeteren (2024-2025),NaN,NaN,True,128282,NaN,8.0,2025-08,1


In [21]:
mean_onderhanden_werk = int(df_features_closed_final['onderhanden_werk_days'].mean())
print(f"Average 'onderhanden werk' time across all features: {mean_onderhanden_werk:.2f} days")

Average 'onderhanden werk' time across all features: 111.00 days


In [22]:
mean_wachttijd = int(df_features_closed_final['waiting_time_days'].mean())
print(f"Average 'wachttijd' time across all features: {mean_wachttijd:.2f} days")

Average 'wachttijd' time across all features: 74.00 days


In [23]:
total_effort_per_domeinteam = df_features_closed_final.groupby('Area')['Effort'].sum().sort_values(ascending=False)
print(total_effort_per_domeinteam)

KeyError: 'Area'

## 6. Check the final DataFrame with all relevant information for each feature + export
List of all features ordered by closed date, with the following columns:
- Epic Name
- Feature Name
- Feature ID
- Area
- Effort (Moeite Int)
- Business Value
- Closing Date
- Wachttijd_feature (waiting time from prioritized to in progress)
- Onder_handen_werk_feature (time from in progress to closed)

### 6.1 Final checks and cleaning

In [ ]:
# Display the final dataframe with waiting time and onderhanden werk days
df_features_closed_final.head(10)

In [ ]:
# Remove unnecessary columns for clarity
df_features_closed_final = df_features_closed_final.drop(columns=['type', 'state', 'assigned_to', 'priority', 'value_area', 'description', 'start_date','target_date', 'work_item_id'])
df_features_closed_final.head(10)

In [ ]:
# Rename columns for clarity
df_features_closed_final = df_features_closed_final.rename(columns={
    'id': 'Feature id',
    'title': 'Feature naam',
    'created_date': 'Aangemaakt op',
    'closed_date': 'Gesloten op',
    'moeite_int': 'Effort',
    'area_path': 'Area',
    'business_value': 'Business value',
    'children_count': 'Aantal user stories',
    'parent_id': 'Epic id',
    'parent_title': 'Epic naam',
    'waiting_time_days': 'Wachttijd (dagen)',
    'onderhanden_werk_days': 'Onderhanden werk (dagen)',
    'year_month': 'Jaar-Maand',
    'closed_effort_per_year_month': 'Gesloten effort per jaar-maand'
})
df_features_closed_final.head(10)

### 6.2 Export

In [ ]:
# Export the final dataframe to a XLSX file
output_file = Path(f'../azure_devops_data/features_analysis_{today}.xlsx')
df_features_closed_final.to_excel(output_file, index=False)